# Introduction to NLP Fundamentals


In [1]:
!wget https://raw.githubusercontent.com/umerkang66/ai-ml-dl/refs/heads/master/tensorflow-bootcamp/03-computer-vision-tf/helper_functions.py

--2026-08-22 14:18:03--  https://raw.githubusercontent.com/umerkang66/ai-ml-dl/refs/heads/master/tensorflow-bootcamp/03-computer-vision-tf/helper_functions.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10836 (11K) [text/plain]
Saving to: ‘helper_functions.py’

helper_functions.py 100%[===================>]  10.58K  --.-KB/s    in 0s      

2026-08-22 14:18:03 (79.7 MB/s) - ‘helper_functions.py’ saved [10836/10836]



In [2]:
from helper_functions import unzip_data, plot_loss_curves, compare_historys

## Kaggle Intro to NLP Dataset


In [3]:
!wget https://storage.googleapis.com/ztm_tf_course/nlp_getting_started.zip

--2026-08-22 14:18:07--  https://storage.googleapis.com/ztm_tf_course/nlp_getting_started.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 34.153.2.27, 34.144.171.27, 34.3.0.27, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|34.153.2.27|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 607343 (593K) [application/zip]
Saving to: ‘nlp_getting_started.zip’

nlp_getting_started 100%[===================>] 593.11K  --.-KB/s    in 0.005s  

2026-08-22 14:18:08 (113 MB/s) - ‘nlp_getting_started.zip’ saved [607343/607343]



In [4]:
unzip_data("nlp_getting_started.zip")

## Read the Data


In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split

train_df = pd.read_csv("train.csv")[["text", "target"]]

train_df, val_df = train_test_split(train_df, test_size=0.1, random_state=42)

test_df = pd.read_csv("test.csv")[["text"]]

In [6]:
train_df.head()

,text,target
4620,'McFadden Reportedly to Test Hamstring Thursda...,0
2858,w--=-=-=-[ NEMA warns Nigerians to prepare for...,1
3098,When I was cooking earlier I got electrocuted ...,0
3751,I'm On Fire. http://t.co/WATsmxYTVa,0
5285,More than 40 families affected by the fatal ou...,1


## Shuffle the training data


In [7]:
train_df_shuffled = train_df.sample(frac=1, random_state=42)

In [8]:
train_df_shuffled.head()

,text,target
3347,78 passengers evacuated safely after Green Lin...,1
6177,@KIRO7Seattle Just saw a Bomb Squad car headin...,1
6970,@Kamunt Holy crap it's been forever since I sa...,0
1497,Learning from the Legacy of a Catastrophic Eru...,1
3453,luke + microphone = exploded ovaries,0


In [9]:
val_df.head()

,text,target
2644,So you have a new weapon that can cause un-ima...,1
2227,The f$&amp;@ing things I do for #GISHWHES Just...,0
5448,DT @georgegalloway: RT @Galloway4Mayor: ÛÏThe...,1
132,Aftershock back to school kick off was great. ...,0
6845,in response to trauma Children of Addicts deve...,0


In [10]:
test_df.head()

,text
0,Just happened a terrible car crash
1,"Heard about #earthquake is different cities, s..."
2,"there is a forest fire at spot pond, geese are..."
3,Apocalypse lighting. #Spokane #wildfires
4,Typhoon Soudelor kills 28 in China and Taiwan


In [11]:
train_df.target.value_counts()

,count
target,
0,3916
1,2935


In [12]:
len(train_df), len(test_df)

(6851, 3263)

## Let's visualize some random samples


In [13]:
import random

random_index = random.randint(0, len(train_df) - 1)

for _, row in train_df.iloc[random_index : random_index + 5].iterrows():
    text, target = row

    print(
        f"Target: {target}", "(real disaster)" if target > 0 else "(not real disaster)"
    )
    print(f"Text: {text}\n")

Target: 0 (not real disaster)
Text: Download @ iTunes http://t.co/ocojPPnRh1 'Floods Of Glory' by Luiz Santos #jazz #art #Music

Target: 1 (real disaster)
Text: @reriellechan HE WAS THE LICH KING'S FIRST CASUALTY BLOCK ME BACK I HATE YOU! http://t.co/0Gidg9U45J

Target: 1 (real disaster)
Text: During the 1960s the oryx a symbol of the Arabian Peninsula were annihilated by hunters. 
http://t.co/yangEQBUQW http://t.co/jQ2eH5KGLt

Target: 1 (real disaster)
Text: Next May I'll be free...from school from obligations like family.... Best of all that damn curfew...

Target: 0 (not real disaster)
Text: Pandemonium In Aba As Woman Delivers Baby Without Face (Photos) @... http://t.co/JbxBi93CLu



## Converting text into numbers


In [14]:
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization

In [15]:
average_max_length = round(
    sum([len(i.split()) for i in train_df["text"]]) / len(train_df)
)

In [16]:
# we have already defaulted this, but here we are setting again
max_length = average_max_length  # how many words from a tweet our model will see


text_vectorizer = TextVectorization(
    max_tokens=10000,  # None = no limit on the number of tokens vocab can have
    standardize="lower_and_strip_punctuation",
    split="whitespace",
    ngrams=None,  # treat each word as a token
    output_mode="int",  # convert tokens into integers
    output_sequence_length=max_length,  # pad all outputs to be max_length tokens long
    pad_to_max_tokens=True,
)

In [17]:
# fit the text vectorizer to the training text

text_vectorizer.adapt(train_df["text"])

In [18]:
# create a sample text and pass it through our text vectorizer instance

sample_text = "There's a flood in my street!"

text_vectorizer([sample_text])

<tf.Tensor: shape=(1, 15), dtype=int64, numpy=
array([[282,   3, 206,   4,  13, 674,   0,   0,   0,   0,   0,   0,   0,
          0,   0]])>

In [19]:
# choose a random text from the training dataset and pass it through the text vectorizer

import random

random_sentence = random.choice(train_df["text"])

print(f"Original text:\n{random_sentence} \n")
print(f"Vectorized version:\n{text_vectorizer([random_sentence])}")

Original text:
FedEx will no longer transport bioterror pathogens in wake of anthrax lab mishaps http://t.co/lHpgxc4b8J 

Vectorized version:
[[ 465   42   41  513  756  495 1500    4  616    6  915  653  937    1
     0]]


In [20]:
# get the unique words in the vocabulary of our text vectorizer instance

words_in_vocab = text_vectorizer.get_vocabulary()

print("Top 5 words in vocab: ", words_in_vocab[:5])
print("Bottom 5 words in vocab: ", words_in_vocab[-5:])
print("Number of words in vocab: ", len(words_in_vocab))

Top 5 words in vocab:  ['', '[UNK]', np.str_('the'), np.str_('a'), np.str_('in')]
Bottom 5 words in vocab:  [np.str_('pakthey'), np.str_('pakistan\x89Ûªs'), np.str_('pakistans'), np.str_('pajamas'), np.str_('paints')]
Number of words in vocab:  10000


In [21]:
from tensorflow.keras.layers import Embedding

embedding = Embedding(
    input_dim=len(
        words_in_vocab
    ),  # total vocabulary size (i.e. number of unique tokens in the text)
    output_dim=128,  # set size of embedding vector
    input_length=max_length,  # how long is each input
)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [22]:
random_sentence = random.choice(train_df["text"])


print(f"Original text:\n{random_sentence} \n")
print(f"Vectorized version:\n{text_vectorizer([random_sentence])}")

sample_embed = embedding(text_vectorizer([random_sentence]))


print(f"Embedded version:\n{sample_embed}")
print(f"Embedded version shape:\n{sample_embed.shape}")

Original text:
Panic at the disco te amo 

Vectorized version:
[[ 311   17    2 1689 4447    1    0    0    0    0    0    0    0    0
     0]]
Embedded version:
[[[ 0.00595292  0.01559434 -0.03185278 ... -0.04675815 -0.00430567
    0.02866533]
  [-0.02373909  0.02663222  0.04030237 ... -0.01082014 -0.02513291
    0.01389343]
  [ 0.00245049  0.04021328  0.01500607 ...  0.01818763 -0.0157031
   -0.01652863]
  ...
  [ 0.01537756 -0.00908221 -0.00966275 ...  0.00296547 -0.03890146
   -0.02317268]
  [ 0.01537756 -0.00908221 -0.00966275 ...  0.00296547 -0.03890146
   -0.02317268]
  [ 0.01537756 -0.00908221 -0.00966275 ...  0.00296547 -0.03890146
   -0.02317268]]]
Embedded version shape:
(1, 15, 128)


In [23]:
sample_embed[0][0], sample_embed[0][0].shape

(<tf.Tensor: shape=(128,), dtype=float32, numpy=
 array([ 5.95291704e-03,  1.55943371e-02, -3.18527818e-02, -3.28084119e-02,
        -4.95633371e-02,  3.69116776e-02,  1.01783648e-02,  3.52085568e-02,
         3.02716829e-02, -4.52336185e-02,  1.61737092e-02,  6.36508316e-03,
         2.16636993e-02, -1.55982375e-02, -2.43693832e-02,  1.77009441e-02,
         1.18886605e-02, -4.16339561e-03, -3.82241122e-02,  3.37498263e-03,
         1.66966431e-02, -2.81661749e-03, -1.94578655e-02,  4.73147519e-02,
         4.62138318e-02,  6.16539642e-03,  7.18375295e-03,  2.70057842e-03,
         2.29273774e-02, -2.17672233e-02,  3.61849926e-02,  3.02583836e-02,
        -2.81157382e-02,  3.26134451e-02, -3.95937786e-02, -2.02277191e-02,
         2.72058733e-02,  1.44975819e-02, -4.24027443e-04, -1.75278559e-02,
        -1.81461573e-02, -1.88826676e-02, -2.60172617e-02,  4.30073254e-02,
        -1.24150142e-02, -3.88708338e-02,  4.51578610e-02,  3.61060239e-02,
        -3.14116478e-04,  4.83618043e-0

## Experiment Models

- Model 1: Feed Forward Neural Network (dense model)
- Model 2: LSTM model (RNN)
- Model 3: GRU model (RNN)
- Model 4: Bidirectional-LSTM (RNN)
- Model 5: 1D Convolutional Neural Network (CNN)
- Model 6: Transfer Learning for NLP


## MODEL_0: BaseLine Naive Bayes


In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

# Create tokenization and modelling pipeline
model_0 = Pipeline(
    [
        ("tfidf", TfidfVectorizer()),  # convert words to numbers using tfidf
        ("clf", MultinomialNB()),  # model the text
    ]
)

# Fit the pipeline to the training data
model_0.fit(train_df["text"], train_df["target"])

Pipeline(steps=[('tfidf', TfidfVectorizer()), ('clf', MultinomialNB())])

In [25]:
# Evaluate baseline model
baseline_score = model_0.score(val_df["text"], val_df["target"])
print(f"Our baseline model achieves an accuracy of: {baseline_score * 100:.2f}%")

Our baseline model achieves an accuracy of: 77.82%


In [28]:
# Make predictions
baseline_preds = model_0.predict(val_df["text"])

In [29]:
# Calculate baseline results
from helper_functions import calculate_results

baseline_results = calculate_results(y_true=val_df["target"], y_pred=baseline_preds)
baseline_results

{'accuracy': 77.82152230971128,
 'precision': 0.792992256322435,
 'recall': 0.7782152230971129,
 'f1': 0.7703527809038112}

## MODEL_1: FFNN


In [49]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Embedding, GlobalAveragePooling1D

In [67]:
inputs = tf.keras.layers.Input(
    shape=(1,), dtype=tf.string
)  # inputs are 1 dimenional strings (i.e. sentences)

x = text_vectorizer(inputs)  # turn the input text into numbers
x = embedding(x)  # create an embedding of the numberized inputs

x = GlobalAveragePooling1D()(x)

outputs = Dense(1, activation="sigmoid")(x)


model_1 = tf.keras.Model(inputs, outputs, name="model_1_dense")

In [68]:
model_1.summary()

Model: "model_1_dense"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_10 (InputLayer)     │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization              │ (None, 15)             │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 15, 128)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_6      │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,280,129 (4.88 MB)

 Trainable params: 1,280,129 (4.88 MB)

 Non-trainable params: 0 (0.00 B)

In [69]:
model_1.compile(
    loss="binary_crossentropy",
    optimizer=tf.keras.optimizers.Adam(),
    metrics=["accuracy"],
)

model_1_history = model_1.fit(
    train_df["text"].to_numpy(),
    train_df["target"].to_numpy(),
    epochs=5,
    validation_data=(val_df["text"].to_numpy(), val_df["target"].to_numpy()),
)

Epoch 1/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8641 - loss: 0.4807 - val_accuracy: 0.7900 - val_loss: 0.4976
Epoch 2/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9219 - loss: 0.2836 - val_accuracy: 0.7874 - val_loss: 0.4812
Epoch 3/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9394 - loss: 0.2096 - val_accuracy: 0.7756 - val_loss: 0.5015
Epoch 4/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9520 - loss: 0.1692 - val_accuracy: 0.7808 - val_loss: 0.5301
Epoch 5/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9587 - loss: 0.1428 - val_accuracy: 0.7664 - val_loss: 0.5601


In [73]:
model1_pred_probs = model_1.predict(val_df["text"].to_numpy())

24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


In [75]:
model1_preds = tf.squeeze(tf.round(model1_pred_probs))

model1_preds[:10]

<tf.Tensor: shape=(10,), dtype=float32, numpy=array([0., 0., 0., 0., 1., 0., 0., 0., 0., 1.], dtype=float32)>

In [77]:
model_1_results = calculate_results(y_true=val_df["target"], y_pred=model1_preds)

model_1_results

{'accuracy': 76.64041994750657,
 'precision': 0.7662974449994261,
 'recall': 0.7664041994750657,
 'f1': 0.7643905710524355}

## MODEL2: LSTM


In [87]:
from tensorflow.keras.layers import LSTM

inputs = tf.keras.layers.Input(shape=(1,), dtype=tf.string)

x = text_vectorizer(inputs)
x = embedding(x)
x = LSTM(64, return_sequences=True)(x)
x = LSTM(64)(x)
outputs = Dense(1, activation="sigmoid")(x)

model_2 = tf.keras.Model(inputs, outputs, name="model_2_LSTM")

model_2.summary()

Model: "model_2_LSTM"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_14 (InputLayer)     │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization              │ (None, 15)             │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 15, 128)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, 15, 64)         │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,362,497 (5.20 MB)

 Trainable params: 1,362,497 (5.20 MB)

 Non-trainable params: 0 (0.00 B)

In [88]:
# Compile model
model_2.compile(
    loss="binary_crossentropy",
    optimizer=tf.keras.optimizers.Adam(),
    metrics=["accuracy"],
)

# Fit model
model_2_history = model_2.fit(
    train_df["text"].to_numpy(),
    train_df["target"].to_numpy(),
    epochs=5,
    validation_data=(val_df["text"].to_numpy(), val_df["target"].to_numpy()),
)

Epoch 1/5


215/215 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.9667 - loss: 0.0856 - val_accuracy: 0.7270 - val_loss: 1.4782
Epoch 2/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9809 - loss: 0.0422 - val_accuracy: 0.7336 - val_loss: 1.3587
Epoch 3/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9810 - loss: 0.0377 - val_accuracy: 0.7520 - val_loss: 1.5428
Epoch 4/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.9823 - loss: 0.0349 - val_accuracy: 0.7520 - val_loss: 1.6085
Epoch 5/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.9796 - loss: 0.0451 - val_accuracy: 0.7270 - val_loss: 1.3341


In [89]:
# Make predictions on the validation set
model_2_pred_probs = model_2.predict(val_df["text"].to_numpy())

# Convert prediction probabilities to 0 or 1 binary classes
model_2_preds = tf.squeeze(tf.round(model_2_pred_probs))

# Calculate evaluation metrics (Accuracy, Precision,Recall, F1-score)
from helper_functions import calculate_results

model_2_results = calculate_results(
    y_true=val_df["target"].to_numpy(), y_pred=model_2_preds
)
model_2_results

24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step


{'accuracy': 72.70341207349081,
 'precision': 0.7274018821246452,
 'recall': 0.7270341207349081,
 'f1': 0.7271979246425836}

## MODEL2: GRU


In [91]:
from tensorflow.keras.layers import GRU

inputs = tf.keras.layers.Input(shape=(1,), dtype=tf.string)

x = text_vectorizer(inputs)
x = embedding(x)
x = GRU(64, return_sequences=True)(x)
x = GRU(64)(x)
outputs = Dense(1, activation="sigmoid")(x)

model_3 = tf.keras.Model(inputs, outputs, name="model_3_GRU")

model_3.summary()

Model: "model_3_GRU"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_16 (InputLayer)     │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization              │ (None, 15)             │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 15, 128)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_2 (GRU)                     │ (None, 15, 64)         │        37,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_3 (GRU)                     │ (None, 64)             │        24,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_24 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,342,273 (5.12 MB)

 Trainable params: 1,342,273 (5.12 MB)

 Non-trainable params: 0 (0.00 B)

In [92]:
# Compile model
model_3.compile(
    loss="binary_crossentropy",
    optimizer=tf.keras.optimizers.Adam(),
    metrics=["accuracy"],
)

# Fit model
model_3_history = model_3.fit(
    train_df["text"].to_numpy(),
    train_df["target"].to_numpy(),
    epochs=5,
    validation_data=(val_df["text"].to_numpy(), val_df["target"].to_numpy()),
)

Epoch 1/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.9604 - loss: 0.1074 - val_accuracy: 0.7467 - val_loss: 0.9176
Epoch 2/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.9809 - loss: 0.0495 - val_accuracy: 0.7454 - val_loss: 0.9572
Epoch 3/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.9791 - loss: 0.0447 - val_accuracy: 0.7415 - val_loss: 1.2042
Epoch 4/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.9815 - loss: 0.0368 - val_accuracy: 0.7388 - val_loss: 0.9652
Epoch 5/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.9816 - loss: 0.0376 - val_accuracy: 0.7375 - val_loss: 1.5309


In [94]:
# Make predictions on the validation set
model_3_pred_probs = model_3.predict(val_df["text"].to_numpy())

# Convert prediction probabilities to 0 or 1 binary classes
model_3_preds = tf.squeeze(tf.round(model_3_pred_probs))

# Calculate evaluation metrics (Accuracy, Precision,Recall, F1-score)
from helper_functions import calculate_results

model_3_results = calculate_results(
    y_true=val_df["target"].to_numpy(), y_pred=model_3_preds
)
model_3_results

24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step


{'accuracy': 73.75328083989501,
 'precision': 0.739263578748887,
 'recall': 0.7375328083989501,
 'f1': 0.7380731047208172}